# Phase 5 — NB1: Stage 1 Category Detection (SemEval 2014)

**Goal:** Train Stage 1 Category Detection with two configs:
1. **ASL:** `stage1_2014.yaml` — Asymmetric Loss + pos_weight
2. **Cat-Aware:** `stage1_2014_cataware.yaml` — Cat-Aware Attention

**Dataset:** SemEval 2014 Task 4 Restaurant (5 categories)
- Train: 3,044 sentences / 3,516 opinions (after drop conflict + dedup)
- Test: 800 sentences / 973 opinions

**Input:** `lcminhc/semeval-2014-absa-restaurant` (2014 XMLs)

**Output:** Upload `/kaggle/working/outputs_p5_nb1/` as Kaggle dataset `p5-nb1-stage1`
- `stage1_2014_best.pt`, `stage1_2014_cataware_best.pt`
- `category_detection.jsonl`, `sentiment_records.jsonl`
- Training logs

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml iterative-stratification

In [ ]:
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Wire SemEval 2014 XML
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-2014-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-2014-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'Dataset semeval-2014-absa-restaurant not found'
print(f'Input: {KAGGLE_INPUT}')
print('Files:', os.listdir(KAGGLE_INPUT))

os.makedirs('SemEval-2014', exist_ok=True)
shutil.copy(f'{KAGGLE_INPUT}/Restaurants_Train.xml',
            'SemEval-2014/Restaurants_Train.xml')
shutil.copy(f'{KAGGLE_INPUT}/Restaurants_Test_Gold.xml',
            'SemEval-2014/Restaurants_Test_Gold.xml')
print('SemEval 2014 XML files wired.')

## 1. Prepare Data

In [ ]:
!python scripts/01_prepare_data.py

In [ ]:
for f in ['category_detection.jsonl', 'sentiment_records.jsonl',
         'classification.jsonl', 'contrastive_triplets.jsonl']:
    path = f'data/processed/{f}'
    if os.path.exists(path):
        with open(path) as fp:
            n = sum(1 for _ in fp)
        print(f'{f}: {n} records')
    else:
        print(f'MISSING: {path}')

In [ ]:
import torch, gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Train — ASL Config

Config: `stage1_2014.yaml`
- 5 categories, ASL loss (gamma_neg=2, margin=0.05)
- encoder_lr=2e-5, head_lr=1e-4, 20 epochs, patience=5

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04a_train_stage1.py --config configs/stage1_2014.yaml

In [ ]:
log_path = 'logs/stage1_2014_training.jsonl'
print('=== ASL Training Log ===')
if os.path.exists(log_path):
    print(f'{"Epoch":<8} {"Loss":<12} {"Cat F1":<10} {"Cat P":<10} {"Cat R":<10}')
    print('-' * 52)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<8} {r['train_loss']:<12.4f} {r['category_f1']:<10.4f} "
                  f"{r.get('category_precision', 0):<10.4f} {r.get('category_recall', 0):<10.4f}")
else:
    print('No log found.')

In [ ]:
# Save ASL checkpoint immediately
output_dir = '/kaggle/working/outputs_p5_nb1'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)
src = 'checkpoints/stage1_2014/best.pt'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/stage1_2014_best.pt')
    print(f'stage1_2014_best.pt: {os.path.getsize(src)/1e6:.1f} MB')
if os.path.exists('logs/stage1_2014_training.jsonl'):
    shutil.copy('logs/stage1_2014_training.jsonl', f'{output_dir}/logs/')
    print('ASL log saved')

## 3. Train — Cat-Aware Attention

Config: `stage1_2014_cataware.yaml`
- 5 categories, Cat-Aware Attention
- encoder_lr=1e-5, head_lr=5e-4, 30 epochs, patience=5

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04a_train_stage1.py --config configs/stage1_2014_cataware.yaml

In [ ]:
log_path = 'logs/stage1_2014_cataware_training.jsonl'
print('=== Cat-Aware Training Log ===')
if os.path.exists(log_path):
    print(f'{"Epoch":<8} {"Loss":<12} {"Cat F1":<10} {"Cat P":<10} {"Cat R":<10}')
    print('-' * 52)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<8} {r['train_loss']:<12.4f} {r['category_f1']:<10.4f} "
                  f"{r.get('category_precision', 0):<10.4f} {r.get('category_recall', 0):<10.4f}")
else:
    print('No log found.')

## 4. Results Comparison

In [ ]:
def best_epoch(log_path):
    if not os.path.exists(log_path):
        return None
    best = None
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            if best is None or r['category_f1'] > best['category_f1']:
                best = r
    return best

asl = best_epoch('logs/stage1_2014_training.jsonl')
cat = best_epoch('logs/stage1_2014_cataware_training.jsonl')

print(f'{"Config":<20} {"Best Ep":<10} {"Cat F1":<10} {"Cat P":<10} {"Cat R":<10}')
print('-' * 62)
if asl:
    print(f'{"ASL":<20} {asl["epoch"]:<10} {asl["category_f1"]:<10.4f} '
          f'{asl.get("category_precision", 0):<10.4f} {asl.get("category_recall", 0):<10.4f}')
if cat:
    print(f'{"Cat-Aware":<20} {cat["epoch"]:<10} {cat["category_f1"]:<10.4f} '
          f'{cat.get("category_precision", 0):<10.4f} {cat.get("category_recall", 0):<10.4f}')

if asl and cat:
    if asl['category_f1'] >= cat['category_f1']:
        print(f'\nBest: ASL (Cat F1={asl["category_f1"]:.4f})')
        print('Use stage1_2014_best.pt for NB3')
    else:
        print(f'\nBest: Cat-Aware (Cat F1={cat["category_f1"]:.4f})')
        print('Use stage1_2014_cataware_best.pt for NB3')

## 5. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb1'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

# Checkpoints
for cfg_name, ckpt_dir in [('stage1_2014', 'stage1_2014'),
                             ('stage1_2014_cataware', 'stage1_2014_cataware')]:
    src = f'checkpoints/{ckpt_dir}/best.pt'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{cfg_name}_best.pt')
        print(f'{cfg_name}_best.pt: {os.path.getsize(src)/1e6:.1f} MB')
    else:
        print(f'WARNING: {src} not found')

# Processed data
for fname in ['category_detection.jsonl', 'sentiment_records.jsonl',
              'classification.jsonl', 'contrastive_triplets.jsonl']:
    src = f'data/processed/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{fname}')
        print(f'{fname} copied')

# Logs
for log in ['stage1_2014_training.jsonl', 'stage1_2014_cataware_training.jsonl']:
    src = f'logs/{log}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/logs/{log}')

print(f'\nOutputs saved to {output_dir}')
print('Upload as Kaggle dataset: p5-nb1-stage1')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_nb1_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_nb1')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_nb1_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')